In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , cast, Any
from IPython.display import Image , display , Markdown

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",)

In [ ]:
class blogState(TypedDict):
    topic: str
    outline: str
    blog_post: str

In [ ]:
def create_outline(state: blogState) -> blogState:
    system_prompt = """
    You are an expert content strategist.

    Your task is to generate a detailed, well-structured blog post outline for the given topic.

    Requirements:
    - Return ONLY the outline.
    - Do not write introductions, explanations, notes, or commentary.
    - Use a logical hierarchy with H1, H2, and H3 sections.
    - Cover beginner, intermediate, and advanced aspects when relevant.
    - Include key concepts, practical examples, common mistakes, best practices, and a conclusion section.
    - Ensure the outline is comprehensive enough for another LLM to generate a complete high-quality article from it.
    - Output in clean Markdown format.

    """
    result = model.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=state["topic"])
        ])    
    state["outline"] = result.content[0]['text'] # type: ignore
    return state

In [ ]:
def create_blog_post(state: blogState) -> blogState:
    input = state["outline"]
    system_prompt = "You are a helpful assistant that writes blog posts. Please write a blog post based on the given outline:"
    result = model.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=input)
        ])    
    state["blog_post"] = result.content[0]['text'] # type: ignore
    return state

In [ ]:
graph = StateGraph(blogState)

In [ ]:
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog_post', create_blog_post)

In [ ]:
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog_post')
graph.add_edge('create_blog_post', END)

workflow = graph.compile()

In [ ]:
res = workflow.invoke(cast(Any , {"topic": "The benefits of meditation"}))

display(Markdown(res["outline"]))
display(Markdown(res["blog_post"]))

In [ ]:
display(Markdown(res["outline"]))

In [ ]:
display(Markdown(res["blog_post"]))